### Bài 4: Mini-batch/Stochastic Gradient Descent

Yêu cầu 1: Cập nhật class LinearRegression để hỗ trợ Mini-batch.

In [19]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [14]:
df = pd.read_csv("advertising.csv")
print(df.head())

      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3   12.0
3  151.5   41.3       58.5   16.5
4  180.8   10.8       58.4   17.9


In [15]:
X = df[['TV','Radio','Newspaper']].values
y = df['Sales'].values

scaler = StandardScaler()

X = scaler.fit_transform(X)

In [16]:
class LinearRegression:
    def __init__(self, learning_rate = 0.001, epochs = 1000, batch_size = 32, shuffle=True):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.w = 0.0
        self.b = 0.0
        self.losses = []

    def predict(self, X):
        return X @ self.w + self.b
    
    def loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)
    
    def batch(self, X, y):
        n_samples = X.shape[0]
        indices = np.arange(n_samples)
        
        if self.shuffle :
            np.random.shuffle(indices)
        
        for start in range (0, n_samples, self.batch_size):
            end = start + self.batch_size
            batch_idx = indices[start:end]
            yield X[batch_idx], y[batch_idx]

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0

        for epoch in range(self.epochs):

            for X_batch, y_batch in self.batch(X, y):
                y_pred = self.predict(X_batch)
                error = y_pred - y_batch

                dw = (1 / len(X_batch)) * (X_batch.T @ error)
                db = (1 / len(X_batch)) * np.sum(error)

                self.w -= self.learning_rate * dw
                self.b -= self.learning_rate * db

            y_pred_all = self.predict(X)
            loss_value = self.loss(y, y_pred_all)
            self.losses.append(loss_value)

    def evaluate(self, X, y):
        y_pred = self.predict(X)

        mae = np.mean(np.abs(y - y_pred))
        mse = np.mean((y - y_pred) ** 2)
        rmse = np.sqrt(mse)

        ss_total = np.sum((y - np.mean(y)) ** 2)
        ss_residual = np.sum((y - y_pred) ** 2)
        r2 = 1 - (ss_residual / ss_total)

        return mae, mse, rmse, r2

In [18]:
model = LinearRegression(
    learning_rate=0.001,
    epochs=1000,
    batch_size=32,
    shuffle=True
)

model.fit(X, y)
mae, mse, rmse, r2 = model.evaluate(X, y)

print("w:", model.w)
print("b:", model.b)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

w: [4.65907788 1.57515573 0.01325669]
b: 15.12071884593111
MAE: 1.2365237081626548
RMSE: 1.6450537619292989
R2: 0.9025842441593901


Yêu cầu 2: So sánh hiệu suất giữa các phương pháp.

In [27]:
def run_model(batch_size, name):

    model = LinearRegression(
        learning_rate=0.001,
        epochs=5000,
        batch_size=batch_size,
        shuffle=True
    )

    start = time.time()

    model.fit(X, y)

    end = time.time()

    mae, mse, rmse, r2 = model.evaluate(X, y)

    print(f"\n{name}")
    print("Batch size:", batch_size)
    print("Time:", round(end-start,4),"s")
    print("MAE:", round(mae,4))
    print("RMSE:", round(rmse,4))
    print("R2:", round(r2,4))

    return model

In [28]:
model_bgd = run_model(
    batch_size = len(X),
    name = "Batch Gradient Descent"
)


Batch Gradient Descent
Batch size: 200
Time: 0.1289 s
MAE: 1.2395
RMSE: 1.6488
R2: 0.9021


In [29]:
model_mbgd = run_model(
    batch_size = 32,
    name = "Mini-batch Gradient Descent"
)


Mini-batch Gradient Descent
Batch size: 32
Time: 0.4088 s
MAE: 1.2363
RMSE: 1.645
R2: 0.9026


In [30]:
model_sgd = run_model(
    batch_size = 1,
    name = "Stochastic Gradient Descent"
)


Stochastic Gradient Descent
Batch size: 1
Time: 32.1953 s
MAE: 1.2365
RMSE: 1.645
R2: 0.9026
